# VARC (Vision ARC) — Offline Training + Test-Time Training on Kaggle

This notebook runs the **official, unmodified** code from [lillian039/VARC](https://github.com/lillian039/VARC),
the reference implementation for the paper *["ARC Is a Vision Problem!"](https://arxiv.org/abs/2511.14761)*
(Hu et al.). It:

1. Clones the repo as-is (no code edits — same `ARC_ViT.py`, `ARC_loader.py`, `offline_train_ARC.py`,
   `test_time_train_ARC.py` you already have).
2. Runs **offline training** of the ViT model on the ARC-AGI-1 training set (+ optional RE-ARC data).
3. Builds the augmented per-task **test-time-training (TTT)** dataset and runs TTT + inference on a
   configurable set of ARC-AGI-1 evaluation tasks.
4. Aggregates predictions into Pass@1 / Pass@2 / Oracle scores, the same metrics the repo's own
   `analysis.py` reports.

### About scale
The authors trained on **8×H200 GPUs** (offline: ~5h; TTT: all 400 eval tasks run in parallel across
8 GPUs). Kaggle gives you a **single GPU** (T4/P100, ~9h/session, ~30h/week quota). The code is left
completely untouched — only the **command-line hyperparameters** in the Config cell below are yours to
tune. Everything is exposed there: epochs, batch size, RE-ARC usage, number of TTT tasks, etc.
Defaults are set to the values from the repo's own `script/*.sh` files; lower `OFFLINE_EPOCHS`,
`REARC_LIMIT`, and `NUM_TTT_TASKS` for a quick smoke test.


## 0. Check GPU
Make sure the Kaggle notebook's **Accelerator** is set to a GPU (Settings → Accelerator → GPU T4 x2 / P100).

In [1]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Tue Aug  4 11:34:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone the official VARC repo (unmodified)

The ARC-AGI-1 and RE-ARC raw data ship inside the repo itself, so no separate data download is needed.

In [2]:
import os, subprocess

WORK_DIR = "/kaggle/working"
REPO_DIR = f"{WORK_DIR}/VARC"

os.chdir(WORK_DIR)
if not os.path.isdir(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/lillian039/VARC.git"],
        cwd=WORK_DIR, capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0 or not os.path.isdir(REPO_DIR):
        raise RuntimeError(
            "git clone failed, so the repo was not created. The #1 cause on Kaggle is that "
            "'Internet' is turned off for this notebook. Fix: Notebook Settings (top right, "
            "three-dot menu or side panel) -> Internet -> toggle ON, then re-run this cell. "
            "You may also need to verify your Kaggle account (phone verification) to enable internet access."
        )
else:
    print("Repo already present, skipping clone.")

os.chdir(REPO_DIR)
!ls



Cloning into 'VARC'...
Updating files:  86% (2049/2380)
Updating files:  87% (2071/2380)
Updating files:  88% (2095/2380)
Updating files:  89% (2119/2380)
Updating files:  90% (2142/2380)
Updating files:  91% (2166/2380)
Updating files:  92% (2190/2380)
Updating files:  93% (2214/2380)
Updating files:  93% (2226/2380)
Updating files:  94% (2238/2380)
Updating files:  95% (2261/2380)
Updating files:  96% (2285/2380)
Updating files:  97% (2309/2380)
Updating files:  98% (2333/2380)
Updating files:  99% (2357/2380)
Updating files: 100% (2380/2380)
Updating files: 100% (2380/2380), done.

analysis.py	 LICENSE	       README.md	 src
assets		 offline_train_ARC.py  requirements.txt  test_time_train_ARC.py
augment_data.py  raw_data	       script		 utils


## 2. Install dependencies

Kaggle already ships a CUDA-enabled PyTorch + NumPy, so we deliberately **don't** force the exact
`torch==2.7.0` / `numpy==2.2.6` pins from `requirements.txt` (doing so can break the pre-configured GPU
driver stack). Everything else from `requirements.txt` is installed as-is.

In [3]:
!pip install -q timm==1.0.12 einops huggingface_hub "wandb==0.22.0" diffusers datasets tqdm
print("Done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 79.9 MB/s eta 0:00:00
Done.


## 3. Configuration (edit this cell)

Every hyperparameter used by `offline_train_ARC.py` and `test_time_train_ARC.py` is defined here.
Defaults for the **model / offline-training** block mirror `script/offline_train_VARC_ViT.sh`
(the exact config used to train VARC-ViT-18M in the paper). Defaults for **TTT** mirror
`script/test_time_training_VARC_ViT_ARC1.sh`. Change anything you like — the launch cells below just
forward these variables as CLI flags to the untouched scripts.

In [4]:
import sys, os, subprocess, json, time, glob

os.chdir(REPO_DIR)

# ----------------------------------------------------------------------------------
# Model architecture (must be identical for offline training and TTT — TTT resumes
# from the offline checkpoint, so changing these after offline training breaks loading)
# ----------------------------------------------------------------------------------
ARCHITECTURE   = "vit"     # "vit" or "unet"
IMAGE_SIZE     = 64        # canvas size
PATCH_SIZE     = 2         # ViT patch size
EMBED_DIM      = 512
DEPTH          = 10
NUM_HEADS      = 8
NUM_COLORS     = 12        # 10 ARC colors + background-canvas color + border/shape token

# torch.compile (script/*.sh leave it ON by default, i.e. --no-compile is NOT passed). On a single
# Kaggle GPU it adds extra CUDA-graph memory pools on top of an already-tight budget, and TTT launches
# a fresh model per task so it would recompile from scratch every time. Recommended True on Kaggle;
# set False if you have GPU headroom and want raw throughput on a long offline-training run.
NO_COMPILE = True

# ----------------------------------------------------------------------------------
# Offline training  (script/offline_train_VARC_ViT.sh uses epochs=100 on 8xH200 ~5h12m)
# ----------------------------------------------------------------------------------
OFFLINE_EPOCHS      = 1          # paper default. Try 5-10 for a quick smoke test on Kaggle.
# NOTE: the repo uses batch-size 32 *per GPU* on 80GB H200s. A single Kaggle T4/P100 has ~15GB,
# and at image-size=64/patch-size=2 each sample is a 1025-token sequence, so 32 will OOM on one GPU.
# Start small and raise until you're close to OOM: 8 is safe on a T4, try 12-16 on a P100.
OFFLINE_BATCH_SIZE  = 8
LEARNING_RATE        = 3e-4
WEIGHT_DECAY          = 0
LR_SCHEDULER          = "cosine"
INCLUDE_REARC         = True       # adds RE-ARC synthetic data (400 tasks x up to 1000 examples each)
REARC_LIMIT           = 1         # -1 = all examples/task (paper). e.g. 20-50 for a much faster run.
DATA_ROOT             = "raw_data/ARC-AGI"
TRAIN_SPLIT            = "training"
NUM_WORKERS            = 2
VIS_EVERY              = 50
SAVE_PATH              = "saves/offline_train_ViT/checkpoint_final.pt"
BEST_SAVE_PATH         = "saves/offline_train_ViT/checkpoint_best.pt"

USE_WANDB              = False     # set True + `wandb login` beforehand to enable
WANDB_PROJECT          = "VisionARC"
WANDB_RUN_NAME         = "offline_train_VARC_kaggle"

# ----------------------------------------------------------------------------------
# Test-time training (TTT). script/test_time_training_VARC_ViT_ARC1.sh values below.
# ----------------------------------------------------------------------------------
TTT_EPOCHS         = 50
TTT_BATCH_SIZE     = 8
TTT_NUM_ATTEMPTS   = 10     # random augmented views voted over per test example
TTT_NUM_EACH       = 1      # independent TTT re-runs per task (for ensembling)
TTT_EVAL_SAVE_NAME = "ARC_1_eval_ViT"   # predictions saved under outputs/{name}_attempt_{i}/

# The repo's script runs TTT on ALL 400 ARC-AGI-1 evaluation tasks (in parallel across 8 GPUs).
# On a single Kaggle GPU that is sequential and slow, so by default we only run a handful.
# Bump NUM_TTT_TASKS (up to 400) or replace TTT_TASKS with your own list of task ids to do more.
NUM_TTT_TASKS = 30

ALL_ARC1_EVAL_TASKS = [
    "af24b4cc", "e1d2900e", "903d1b4a", "4e469f39", "b1fc8b8e", "2c737e39", "992798f6", "00576224", "48131b3c", "60a26a3e", 
    "59341089", "31d5ba1a", "e633a9e5", "62ab2642", "73c3b0d8", "c663677b", "c48954c1", "08573cc6", "136b0064", "929ab4e9", 
    "5b526a93", "ef26cbf6", "fafd9572", "67c52801", "ad7e01d0", "506d28a5", "27a77e38", "d492a647", "72a961c9", "fd4b2b02"
]

TTT_TASKS = ALL_ARC1_EVAL_TASKS[:NUM_TTT_TASKS]
print(f"Will run offline training for {OFFLINE_EPOCHS} epochs, then TTT on {len(TTT_TASKS)} task(s):")
print(TTT_TASKS)

Will run offline training for 1 epochs, then TTT on 30 task(s):
['af24b4cc', 'e1d2900e', '903d1b4a', '4e469f39', 'b1fc8b8e', '2c737e39', '992798f6', '00576224', '48131b3c', '60a26a3e', '59341089', '31d5ba1a', 'e633a9e5', '62ab2642', '73c3b0d8', 'c663677b', 'c48954c1', '08573cc6', '136b0064', '929ab4e9', '5b526a93', 'ef26cbf6', 'fafd9572', '67c52801', 'ad7e01d0', '506d28a5', '27a77e38', 'd492a647', '72a961c9', 'fd4b2b02']


## 4. Helper: run a command and stream its output live

In [5]:
import sys

def run_streaming(cmd, cwd=REPO_DIR, env=None):
    """Run `cmd` (a list of args) and stream stdout/stderr to the notebook live,
    including the scripts' \r-updating epoch progress bars (train_loss/train_acc/
    eval_acc/eta etc. get printed once per epoch; the % bar updates continuously
    within an epoch)."""
    print("Running:", " ".join(cmd))
    full_env = os.environ.copy()
    full_env["PYTHONUNBUFFERED"] = "1"   # make the child flush immediately, not just on newline
    if env:
        full_env.update(env)

    proc = subprocess.Popen(
        cmd, cwd=cwd, env=full_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    # Read char-by-char (not line-by-line) so \r progress-bar updates render live
    # instead of being buffered until the next real \n.
    while True:
        ch = proc.stdout.read(1)
        if ch == "" and proc.poll() is not None:
            break
        if ch:
            sys.stdout.write(ch)
            sys.stdout.flush()

    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return proc.returncode


In [6]:
mem_env = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "PYTORCH_ALLOC_CONF": "expandable_segments:True"}

In [7]:
BEST_SAVE_PATH='/kaggle/input/datasets/amritajoshi0720/exp2data/checkpoint_best.pt'

## 6. Build the augmented per-task TTT dataset

Reproduces `augment_data.py` (called exactly the same way, just imported instead of run as `__main__`):
for every ARC-AGI-1 evaluation task it writes an `eval_color_permute_ttt_9/<task_id>/` folder containing
the task's demonstration pairs plus 9 color-permuted copies, which `test_time_train_ARC.py` treats as
the per-task training set for TTT.

In [8]:
from utils.data_augmentation import augment_raw_data_split_per_task

t0 = time.time()
augment_raw_data_split_per_task(
    dataset_root="raw_data/ARC-AGI",
    split="evaluation",
    output_subdir="eval_color_permute_ttt_9",
    num_permuate=9,
    only_basic=True,
)
print(f"Augmented TTT dataset built in {(time.time()-t0)/60:.1f} min")
print("Example folder contents:", os.listdir(f"raw_data/ARC-AGI/data/eval_color_permute_ttt_9/{TTT_TASKS[0]}"))


5 augmenters will be applied
[augment] Completed split 'evaluation': 20000 augmented tasks prepared in raw_data/ARC-AGI/data/eval_color_permute_ttt_9/ff72ca3e
Augmented TTT dataset built in 0.6 min
Example folder contents: ['af24b4cc_rotate_180_perm7_augmented.json', 'af24b4cc_rotate_270_perm8_augmented.json', 'af24b4cc_rotate_90_perm2_augmented.json', 'af24b4cc_rotate_180_perm9_augmented.json', 'af24b4cc_flip_1_augmented.json', 'af24b4cc_rotate_270_augmented.json', 'af24b4cc_flip_0_perm8_augmented.json', 'af24b4cc_rotate_90_perm5_augmented.json', 'af24b4cc_rotate_270_perm2_augmented.json', 'af24b4cc_rotate_90_perm4_augmented.json', 'af24b4cc_flip_1_perm2_augmented.json', 'af24b4cc_rotate_270_perm9_augmented.json', 'af24b4cc_flip_1_perm1_augmented.json', 'af24b4cc_rotate_180_perm3_augmented.json', 'af24b4cc_flip_1_perm8_augmented.json', 'af24b4cc_rotate_90_perm1_augmented.json', 'af24b4cc_rotate_270_perm7_augmented.json', 'af24b4cc_rotate_180_perm2_augmented.json', 'af24b4cc_flip_0_per

In [9]:
import os

TTT_SCRIPT = "test_time_train_ARC.py"  # relative to REPO_DIR (cwd should already be REPO_DIR here)
with open(TTT_SCRIPT) as f:
    src = f.read()

# ------------------------------------------------------------------
# 1) init a pixel-accuracy accumulator next to the exact-match ones
# ------------------------------------------------------------------
old_init = """            train_exact = 0
            train_examples = 0"""
new_init = """            train_exact = 0
            train_examples = 0
            train_pixel_correct = 0
            train_pixel_total = 0"""
if new_init in src:
    print("Step 1 already applied, skipping.")
else:
    assert old_init in src, "train_exact/train_examples init block not found"
    src = src.replace(old_init, new_init)

# ------------------------------------------------------------------
# 2) accumulate pixel-level correct/total counts alongside exact-match, per example
# ------------------------------------------------------------------
old_per_example = """                batch_size = inputs.size(0)
                predictions = logits.argmax(dim=1)
                for idx in range(batch_size):
                    target = targets[idx]
                    prediction = predictions[idx]
                    valid = target != IGNORE_INDEX
                    if valid.any():
                        is_exact = bool(torch.equal(prediction[valid], target[valid]))
                    else:
                        is_exact = False
                    train_exact += int(is_exact)
                    train_examples += 1"""
new_per_example = """                batch_size = inputs.size(0)
                predictions = logits.argmax(dim=1)
                for idx in range(batch_size):
                    target = targets[idx]
                    prediction = predictions[idx]
                    valid = target != IGNORE_INDEX
                    if valid.any():
                        is_exact = bool(torch.equal(prediction[valid], target[valid]))
                        train_pixel_correct += int((prediction[valid] == target[valid]).sum().item())
                        train_pixel_total += int(valid.sum().item())
                    else:
                        is_exact = False
                    train_exact += int(is_exact)
                    train_examples += 1"""
if new_per_example in src:
    print("Step 2 already applied, skipping.")
else:
    assert old_per_example in src, "per-example train loop not found"
    src = src.replace(old_per_example, new_per_example)

# ------------------------------------------------------------------
# 3) fold pixel counts into the distributed reduction, compute train_pixel_acc
# ------------------------------------------------------------------
old_reduce = """            train_totals = torch.tensor(
                [running_loss, sample_count, train_exact, train_examples],
                dtype=torch.float64,
                device=device,
            )
            if distributed and dist.is_initialized():
                dist.all_reduce(train_totals, op=dist.ReduceOp.SUM)
            running_loss_total, sample_count_total, train_exact_total, train_examples_total = train_totals.tolist()
            avg_train_loss = running_loss_total / max(sample_count_total, 1)
            train_acc = train_exact_total / max(train_examples_total, 1)"""
new_reduce = """            train_totals = torch.tensor(
                [running_loss, sample_count, train_exact, train_examples, train_pixel_correct, train_pixel_total],
                dtype=torch.float64,
                device=device,
            )
            if distributed and dist.is_initialized():
                dist.all_reduce(train_totals, op=dist.ReduceOp.SUM)
            (running_loss_total, sample_count_total, train_exact_total, train_examples_total,
             train_pixel_correct_total, train_pixel_total_total) = train_totals.tolist()
            avg_train_loss = running_loss_total / max(sample_count_total, 1)
            train_acc = train_exact_total / max(train_examples_total, 1)
            train_pixel_acc = train_pixel_correct_total / max(train_pixel_total_total, 1)"""
if new_reduce in src:
    print("Step 3 already applied, skipping.")
else:
    assert old_reduce in src, "train_totals reduction block not found"
    src = src.replace(old_reduce, new_reduce)

# ------------------------------------------------------------------
# 4) add train_pixel_acc into the per-epoch log line that's already printed live
# ------------------------------------------------------------------
old_log = """            log_parts = [
                f"epoch={epoch}",
                f"train_loss={avg_train_loss:.4f}",
                f"train_acc={train_acc:.4f}",
                f"epoch_time={epoch_duration:.1f}s",
                f"eta_total={_format_eta(total_eta)}",
            ]"""
new_log = """            log_parts = [
                f"epoch={epoch}",
                f"train_loss={avg_train_loss:.4f}",
                f"train_acc={train_acc:.4f}",
                f"train_pixel_acc={train_pixel_acc:.4f}",
                f"epoch_time={epoch_duration:.1f}s",
                f"eta_total={_format_eta(total_eta)}",
            ]"""
if new_log in src:
    print("Step 4 already applied, skipping.")
else:
    assert old_log in src, "log_parts block not found"
    src = src.replace(old_log, new_log)

# ------------------------------------------------------------------
# 5) THIS is the step your earlier patch was missing — write it back to disk
# ------------------------------------------------------------------
with open(TTT_SCRIPT, "w") as f:
    f.write(src)

print(f"✅ Patched {TTT_SCRIPT}: every epoch will now print train_pixel_acc alongside train_loss/train_acc.")

✅ Patched test_time_train_ARC.py: every epoch will now print train_pixel_acc alongside train_loss/train_acc.


In [10]:
with open("test_time_train_ARC.py") as f:
    src = f.read()

FREEZE_EXCEPT_TASK_TOKEN = True   # True = only the task-embedding vector is updated during TTT
                                    # False = normal VARC behavior (fine-tune all ~18M params)

freeze_marker = "# === injected: freeze everything except task-embedding params for TTT ==="

if freeze_marker in src:
    print("Freeze patch already applied, skipping." if FREEZE_EXCEPT_TASK_TOKEN
          else "Freeze patch is applied but FREEZE_EXCEPT_TASK_TOKEN=False — "
               "the injected code stays but is a no-op only if you also flip it below.")
elif FREEZE_EXCEPT_TASK_TOKEN:
    old = """    optimizer, scaler, scheduler = load_optimizer(
        model=model, args=args, device=device, distributed=distributed, rank=rank
    )"""
    new = """    %s
    _task_param_names = [n for n, _ in model.named_parameters() if "task" in n.lower()]
    if not _task_param_names:
        raise RuntimeError('No task-embedding parameters found (looked for "task" in param name).')
    for _n, _p in model.named_parameters():
        _p.requires_grad = ("task" in _n.lower())
    _trainable = sum(_p.numel() for _p in model.parameters() if _p.requires_grad)
    _total = sum(_p.numel() for _p in model.parameters())
    print(f"[TTT] training ONLY task-embedding params: {_task_param_names}")
    print(f"[TTT] trainable params: {_trainable} / {_total} ({100*_trainable/_total:.4f}%%)")
    # === end injected block ===

    optimizer, scaler, scheduler = load_optimizer(
        model=model, args=args, device=device, distributed=distributed, rank=rank
    )""" % freeze_marker

    assert old in src, "Expected load_optimizer call not found — VARC source may have changed."
    src = src.replace(old, new)

    with open("test_time_train_ARC.py", "w") as f:
        f.write(src)
    print("✅ Patched: TTT will only update task-embedding parameters (backbone frozen).")
else:
    print("Freeze disabled — TTT will fine-tune all model parameters (default VARC behavior).")

✅ Patched: TTT will only update task-embedding parameters (backbone frozen).


## 7. Test-time training (TTT)

For each task in `TTT_TASKS`, calls `test_time_train_ARC.py` exactly as
`script/test_time_training_VARC_ViT_ARC1.sh` does per task (minus the outer GPU-parallel loop, since
we have one GPU). Each call: fine-tunes a fresh copy of the offline-trained model on that task's
augmented demonstrations (`TTT_NUM_EACH` independent times), then predicts the held-out test pair(s)
with `TTT_NUM_ATTEMPTS` augmented views per attempt and majority-votes the answer. Predictions and a
per-task Pass@1/Pass@2/Oracle report (via the repo's own `analyze_prediction.py`) are printed live and
saved under `outputs/{TTT_EVAL_SAVE_NAME}_attempt_{0..TTT_NUM_EACH-1}/`.

In [11]:
def build_ttt_cmd(task_name):
    return [
        sys.executable, "test_time_train_ARC.py",
        "--epochs", str(TTT_EPOCHS),
        "--depth", str(DEPTH),
        "--batch-size", str(TTT_BATCH_SIZE),
        "--image-size", str(IMAGE_SIZE),
        "--patch-size", str(PATCH_SIZE),
        "--learning-rate", str(LEARNING_RATE),
        "--weight-decay", str(WEIGHT_DECAY),
        "--embed-dim", str(EMBED_DIM),
        "--num-heads", str(NUM_HEADS),
        "--num-colors", str(NUM_COLORS),
        "--resume-checkpoint", BEST_SAVE_PATH,
        "--resume-skip-task-token",
        "--lr-scheduler", LR_SCHEDULER,
        "--train-split", f"eval_color_permute_ttt_9/{task_name}",
        "--eval-split", f"eval_color_permute_ttt_9/{task_name}",
        "--data-root", DATA_ROOT,
        "--architecture", ARCHITECTURE,
        "--eval-save-name", TTT_EVAL_SAVE_NAME,
        "--num-attempts", str(TTT_NUM_ATTEMPTS),
        "--ttt-num-each", str(TTT_NUM_EACH),
    ] + (["--no-compile"] if NO_COMPILE else [])

import os, json, time, sys
import numpy as np
from utils.eval_utils import get_majority_vote

def _safe_2d_shape(arr):
    """Return (h, w) only if arr is a proper 2D rectangular grid, else None."""
    if arr is None or getattr(arr, "ndim", None) != 2:
        return None
    return arr.shape

def compute_task_pixel_stats(task_name, save_name, num_each, data_root):
    """Merge predictions across TTT attempts for one task, compute pixel/exact
    stats using the top majority-voted prediction (same guess used for pass@1)."""
    task_type = data_root.split("/")[-1]
    merged = None
    for i in range(num_each):
        fp = f"outputs/{save_name}_attempt_{i}/{task_name}_predictions.json"
        if not os.path.exists(fp):
            continue
        with open(fp) as f:
            d = json.load(f)
        if merged is None:
            merged = {k: list(v) for k, v in d.items()}
        else:
            for k, v in d.items():
                merged.setdefault(k, [])
                merged[k] += v
    if merged is None:
        print(f"  [warn] no predictions found for {task_name}")
        return None

    gt_path = f"raw_data/{task_type}/data/evaluation/{task_name}.json"
    with open(gt_path) as f:
        gt_data = json.load(f)

    correct_px = total_px = exact = pairs = 0
    rows = []
    for idx_str, preds in merged.items():
        pair_idx = int(idx_str)
        gt_grid = np.array(gt_data["test"][pair_idx]["output"])
        mv = get_majority_vote(preds)
        pred_grid = np.array(mv[0]["prediction"]) if mv else None

        if pred_grid is not None and pred_grid.shape == gt_grid.shape:
            correct = int(np.sum(pred_grid == gt_grid))
        elif pred_grid is not None:
            gt_shape = _safe_2d_shape(gt_grid)
            pred_shape = _safe_2d_shape(pred_grid)
            if gt_shape is not None and pred_shape is not None:
                min_h = min(gt_shape[0], pred_shape[0])
                min_w = min(gt_shape[1], pred_shape[1])
                correct = int(np.sum(gt_grid[:min_h, :min_w] == pred_grid[:min_h, :min_w])) if min_h > 0 and min_w > 0 else 0
            else:
                correct = 0  # ragged/malformed prediction, can't compare pixel-wise
        else:
            correct = 0

        total = gt_grid.size
        is_exact = pred_grid is not None and pred_grid.shape == gt_grid.shape and correct == total
        pix_acc = correct / total * 100 if total else 0.0

        correct_px += correct; total_px += total; exact += int(is_exact); pairs += 1
        rows.append((pair_idx, pix_acc, is_exact))

    task_pixel_acc = correct_px / max(total_px, 1) * 100
    task_exact_acc = exact / max(pairs, 1) * 100
    print(f"  -> {task_name}: pixel_acc={task_pixel_acc:.2f}%  exact_match={task_exact_acc:.2f}%  ({pairs} test pair(s))")
    for pair_idx, pix_acc, is_exact in rows:
        print(f"       pair {pair_idx}: pixel_acc={pix_acc:6.2f}%  {'✅' if is_exact else '❌'}")

    return dict(correct_px=correct_px, total_px=total_px, exact=exact, pairs=pairs)

# --- Run TTT task-by-task, printing pixel accuracy right after each ---
t0 = time.time()
task_pixel_stats = {}
for i, task_name in enumerate(TTT_TASKS, 1):
    print(f"\n===== [{i}/{len(TTT_TASKS)}] TTT on task {task_name} =====")
    run_streaming(build_ttt_cmd(task_name), env=mem_env)
    stats = compute_task_pixel_stats(task_name, TTT_EVAL_SAVE_NAME, TTT_NUM_EACH, DATA_ROOT)
    if stats is not None:
        task_pixel_stats[task_name] = stats
print(f"\nTTT finished for {len(TTT_TASKS)} task(s) in {(time.time()-t0)/60:.1f} min")



===== [1/30] TTT on task af24b4cc =====
Running: /usr/bin/python3 test_time_train_ARC.py --epochs 50 --depth 10 --batch-size 8 --image-size 64 --patch-size 2 --learning-rate 0.0003 --weight-decay 0 --embed-dim 512 --num-heads 8 --num-colors 12 --resume-checkpoint /kaggle/input/datasets/amritajoshi0720/exp2data/checkpoint_best.pt --resume-skip-task-token --lr-scheduler cosine --train-split eval_color_permute_ttt_9/af24b4cc --eval-split eval_color_permute_ttt_9/af24b4cc --data-root raw_data/ARC-AGI --architecture vit --eval-save-name ARC_1_eval_ViT --num-attempts 10 --ttt-num-each 1 --no-compile
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because 

## 8. Aggregate results

Merges predictions across the `TTT_NUM_EACH` independent TTT runs per task and computes overall
Pass@1 / Pass@2 / Oracle, the same scoring logic as the repo's `utils/analyze_prediction.py` /
`analysis.py` (majority vote via `get_majority_vote`), generalized to whatever `TTT_TASKS` you ran.

In [12]:
import os, json
import numpy as np
from utils.eval_utils import get_majority_vote

def collect_final_results(task_list, save_name, num_each, data_root):
    task_type = data_root.split("/")[-1]
    all_task_num = correct_1 = correct_2 = correct_oracle = 0
    total_correct_px = total_px = total_exact = total_pairs = 0
    per_task_scores = {}

    print(f"{'Task ID':<12} | {'Pass@1':<7} | {'Pass@2':<7} | {'Oracle':<7} | {'PixelAcc':<9} | {'ExactAcc'}")
    print("-" * 72)

    for task_name in task_list:
        merged = None
        for i in range(num_each):
            fp = f"outputs/{save_name}_attempt_{i}/{task_name}_predictions.json"
            if not os.path.exists(fp):
                continue
            with open(fp) as f:
                d = json.load(f)
            if merged is None:
                merged = {k: list(v) for k, v in d.items()}
            else:
                for k, v in d.items():
                    merged.setdefault(k, [])
                    merged[k] += v
        if merged is None:
            print(f"{task_name:<12} | [no predictions found, skipping]")
            continue

        gt_path = f"raw_data/{task_type}/data/evaluation/{task_name}.json"
        with open(gt_path) as f:
            gt_data = json.load(f)
        ground_truth = {str(i): item["output"] for i, item in enumerate(gt_data["test"])}

        n_examples = len(merged)
        t1 = t2 = torac = 0
        task_correct_px = task_total_px = task_exact = 0

        for idx_str, preds in merged.items():
            mv = get_majority_vote(preds)
            gt = ground_truth[str(idx_str)]
            gt_arr = np.array(gt)

            p1 = bool(mv) and mv[0]["prediction"] == gt
            p2 = p1 or (len(mv) > 1 and mv[1]["prediction"] == gt)
            oracle = any(e["prediction"] == gt for e in mv)
            t1 += p1; t2 += p2; torac += oracle

            pred_arr = np.array(mv[0]["prediction"]) if mv else None
            if pred_arr is not None and pred_arr.shape == gt_arr.shape:
                correct = int(np.sum(pred_arr == gt_arr))
            elif pred_arr is not None:
                min_h = min(gt_arr.shape[0], pred_arr.shape[0])
                min_w = min(gt_arr.shape[1], pred_arr.shape[1])
                correct = int(np.sum(gt_arr[:min_h, :min_w] == pred_arr[:min_h, :min_w])) if min_h > 0 and min_w > 0 else 0
            else:
                correct = 0

            task_correct_px += correct
            task_total_px += gt_arr.size
            task_exact += int(p1)

        task_pass1, task_pass2, task_oracle = t1 / max(n_examples, 1), t2 / max(n_examples, 1), torac / max(n_examples, 1)
        task_pixel_acc = task_correct_px / max(task_total_px, 1) * 100
        task_exact_acc = task_exact / max(n_examples, 1) * 100

        per_task_scores[task_name] = dict(pass_at_1=task_pass1, pass_at_2=task_pass2, oracle=task_oracle,
                                           pixel_acc=task_pixel_acc, exact_acc=task_exact_acc)

        all_task_num += 1
        correct_1 += task_pass1; correct_2 += task_pass2; correct_oracle += task_oracle
        total_correct_px += task_correct_px; total_px += task_total_px
        total_exact += task_exact; total_pairs += n_examples

        print(f"{task_name:<12} | {task_pass1:<7.2f} | {task_pass2:<7.2f} | {task_oracle:<7.2f} | {task_pixel_acc:<8.2f}% | {task_exact_acc:.2f}%")

    print("=" * 72)
    if all_task_num:
        overall_pixel_acc = total_correct_px / max(total_px, 1) * 100
        overall_exact_acc = total_exact / max(total_pairs, 1) * 100
        print(f"\n==== Aggregate over {all_task_num} task(s), {total_pairs} test pair(s) ====")
        print(f"Pass@1              : {correct_1/all_task_num:.4f}")
        print(f"Pass@2              : {correct_2/all_task_num:.4f}")
        print(f"Oracle              : {correct_oracle/all_task_num:.4f}")
        print(f"Overall Pixel Acc   : {overall_pixel_acc:.2f}%  ({total_correct_px}/{total_px} pixels)")
        print(f"Overall Exact Match : {overall_exact_acc:.2f}%  ({total_exact}/{total_pairs} pairs)")
    return per_task_scores

final_scores = collect_final_results(TTT_TASKS, TTT_EVAL_SAVE_NAME, TTT_NUM_EACH, DATA_ROOT)

Task ID      | Pass@1  | Pass@2  | Oracle  | PixelAcc  | ExactAcc
------------------------------------------------------------------------
af24b4cc     | 0.00    | 0.00    | 0.00    | 50.00   % | 0.00%
e1d2900e     | 0.00    | 0.00    | 0.00    | 98.33   % | 0.00%
903d1b4a     | 0.00    | 0.00    | 0.00    | 92.58   % | 0.00%
4e469f39     | 0.00    | 0.00    | 0.00    | 80.00   % | 0.00%
b1fc8b8e     | 0.00    | 0.00    | 0.00    | 58.00   % | 0.00%
2c737e39     | 0.00    | 0.00    | 0.00    | 90.91   % | 0.00%
992798f6     | 0.00    | 0.00    | 0.00    | 95.70   % | 0.00%
00576224     | 0.00    | 0.00    | 0.00    | 11.11   % | 0.00%


IndexError: tuple index out of range

# **Visualization of results**
personal visualizer - it can take input the predicted files from above runs and then display input, ground truth and predicted grid together
NOTE - only for personal use and not to be done along with the above model

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from utils.eval_utils import get_majority_vote

arc_cmap = colors.ListedColormap(['#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
                                   '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'])
norm = colors.Normalize(vmin=0, vmax=9)

def plot_grid(ax, grid, title):
    grid = np.array(grid)
    ax.imshow(grid, cmap=arc_cmap, norm=norm)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(np.arange(-0.5, grid.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
    ax.grid(which='minor', color='#555555', linestyle='-', linewidth=1)
    ax.set_xticks([]); ax.set_yticks([])

def plot_ttt_results(task_list=TTT_TASKS, save_name=TTT_EVAL_SAVE_NAME, num_each=TTT_NUM_EACH, data_root=DATA_ROOT):
    task_type = data_root.split("/")[-1]
    for task_id in task_list:
        gt_path = f"raw_data/{task_type}/data/evaluation/{task_id}.json"
        if not os.path.exists(gt_path):
            print(f"✗ ground truth missing for {task_id}: {gt_path}")
            continue
        with open(gt_path) as f:
            gt_data = json.load(f)

        merged = None
        for i in range(num_each):
            fp = f"outputs/{save_name}_attempt_{i}/{task_id}_predictions.json"
            if not os.path.exists(fp):
                continue
            with open(fp) as f:
                d = json.load(f)
            if merged is None:
                merged = {k: list(v) for k, v in d.items()}
            else:
                for k, v in d.items():
                    merged.setdefault(k, [])
                    merged[k] += v
        if merged is None:
            print(f"⚠️ no predictions found for {task_id}")
            continue

        for idx_str, preds in merged.items():
            idx = int(idx_str)
            case = gt_data["test"][idx]
            mv = get_majority_vote(preds)
            pred_grid = mv[0]["prediction"] if mv else np.zeros_like(case["input"]).tolist()
            is_exact = pred_grid == case.get("output")

            fig, axs = plt.subplots(1, 3, figsize=(14, 4))
            status = "MATCH" if is_exact else "MISMATCH"
            fig.suptitle(f"Task ID: {task_id} (Test Instance {idx}) — {status}", fontsize=13, fontweight='bold', y=1.05)
            plot_grid(axs[0], case["input"], "True Input Grid (From Source)")
            plot_grid(axs[1], case.get("output", np.zeros_like(case["input"])), "True Target (From Source)")
            plot_grid(axs[2], pred_grid, "Your Model's Prediction")
            plt.tight_layout(); plt.show()

plot_ttt_results()